# Analisi R-GCN

## Dataset

In [ ]:
import pandas as pd
import torch
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.hpo import hpo_pipeline
import json

In [2]:
from data_loader import AtravelDataset

# path
BASE_PATH = "C:\\Users\\User\\Documents\\anisa\\Progetto_DL\\atravel-materialize"

# Inizializzazione
dataset = AtravelDataset(BASE_PATH)
data = dataset.get_splits()

# Data
train_edges, train_types = data['train']
print(f"Loaded {train_edges.size(1)} triples for training.")

Loaded 65401 triples for training.


### Informazioni sul dataset

In [ ]:
stats = dataset.get_statistics()

df_stats = pd.DataFrame(stats).T 
df_stats.columns = ['Train', 'Valid', 'Test']

avg_degrees = []
for split in df_stats.columns:
    entities = df_stats.loc["Entita'", split]
    triples = df_stats.loc["Triple (Edges)", split]
    avg_degrees.append(f"{triples / entities:.2f}")

df_stats.loc['Grado Medio'] = avg_degrees

print("--- Dataset Overview ---")
print(df_stats)

# Codice LaTeX
latex_code = df_stats.to_latex(
    index=True, 
    caption="Statistiche del dataset", 
    label="tab:dataset_stats",
    column_format="|l|c|c|c|"
)
print("\n--- LaTeX Code ---")
print(latex_code)

--- Dataset Overview ---
                Train Valid  Test
Entita'         29767  4966  8069
Relazioni          25    20    19
Triple (Edges)  65401  3847  7695
Grado Medio      2.20  0.77  0.95

--- LaTeX Code ---
\begin{table}
\caption{Statistiche del dataset}
\label{tab:dataset_stats}
\begin{tabular}{|l|c|c|c|}
\toprule
 & Train & Valid & Test \\
\midrule
Entita' & 29767 & 4966 & 8069 \\
Relazioni & 25 & 20 & 19 \\
Triple (Edges) & 65401 & 3847 & 7695 \\
Grado Medio & 2.20 & 0.77 & 0.95 \\
\bottomrule
\end{tabular}
\end{table}



## Triples factory

In [ ]:
def get_pykeen_triples(dataset_split, entity_to_id=None, relation_to_id=None):
    edge_index, edge_type = dataset_split
    
    triples = torch.stack([
        edge_index[0], # Soggetto
        edge_type,    # Relazione
        edge_index[1]  # Oggetto
    ], dim=1).cpu().numpy().astype(str)
    
    return TriplesFactory.from_labeled_triples(
        triples, 
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=False
    )

training_factory = get_pykeen_triples(dataset.get_splits()['train'])

validation_factory = get_pykeen_triples(
    dataset.get_splits()['valid'],
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

testing_factory = get_pykeen_triples(
    dataset.get_splits()['test'],
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tuning degli iperparametri

In [ ]:
hpo_result = hpo_pipeline(
    training=training_factory,
    validation=validation_factory,
    testing=testing_factory,
    model='RGCN',
    model_kwargs_ranges=dict(
        embedding_dim=dict(type=int, low=64, high=128, q=64), 
    ),
    model_kwargs=dict(
        interaction='distmult', 
    ),
    optimizer='Adam',
    optimizer_kwargs_ranges=dict(
        lr=dict(type=float, low=0.001, high=0.01, log=True), 
    ),
    training_kwargs=dict(
        num_epochs=30,
        batch_size=1024,
    ),
    metric='mrr',  # metrica da ottimizzare
    evaluator_kwargs=dict(
        metrics=['mrr', 'mr', 'hits@k'],
    ),
    n_trials=5, 
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

[I 2025-12-31 11:23:56,729] A new study created in memory with name: no-name-9962c506-adb5-4be0-b425-7b030f531f5e
c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.ReLU'> which is of type type.
  warnings.warn(message)
c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.LeakyReLU'> which is of type type.
  warnings.warn(message)
No random seed is specified. Setting to 762401251.
Layers RGCNLayer(
  (fwd): BasesDecomposition(
    (relation_representations): LowRankRepresentation(
      (base): Embedding(
        (_embeddings): Embedding(5, 409

In [ ]:
trial_data = []
for t in hpo_result.study.trials:
    trial_data.append({
        'trial_number': t.number,
        'value': t.value,  
        **t.params,        
    })

df = pd.DataFrame(trial_data)
print(df)


   trial_number     value  model.embedding_dim  model.num_layers  \
0             0  0.116537                   64                 4   
1             1  0.144997                  128                 2   
2             2  0.117659                   64                 2   
3             3  0.100183                   64                 5   
4             4  0.017983                  128                 2   

   model.use_bias                                 model.activation  \
0           False  <class 'torch.nn.modules.activation.LeakyReLU'>   
1           False  <class 'torch.nn.modules.activation.LeakyReLU'>   
2           False       <class 'torch.nn.modules.activation.ReLU'>   
3           False       <class 'torch.nn.modules.activation.ReLU'>   
4            True       <class 'torch.nn.modules.activation.ReLU'>   

   model.edge_dropout  model.self_loop_dropout model.edge_weighting  \
0                 0.3                      0.3    inverse_in_degree   
1                 0.0       

In [8]:
study = hpo_result.study

In [9]:
best_trial = study.best_trial

In [10]:
best_trial.value

0.14499680697917938

## Replica finale

In [11]:
best_params = best_trial.params
final_result = pipeline(
    training=training_factory,
    testing=testing_factory,
    model='RGCN',
    model_kwargs=dict(
        embedding_dim=best_params['model.embedding_dim'],
        num_layers=best_params['model.num_layers'],
        use_bias=best_params['model.use_bias'],
        activation=best_params['model.activation'],
        edge_dropout=best_params['model.edge_dropout'],
        self_loop_dropout=best_params['model.self_loop_dropout'],
        edge_weighting=best_params['model.edge_weighting'],
        decomposition=best_params['model.decomposition'],
        interaction='distmult',
    ),
    optimizer='Adam',
    optimizer_kwargs=dict(
        lr=best_params['optimizer.lr'],
    ),
    training_kwargs=dict(
        num_epochs=30,
        batch_size=1024
    ),
    evaluator='rankbased',
    evaluator_kwargs=dict(
        metrics=['mrr', 'mr', 'hits@k'],
    ),
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=True for RGCNRepresentation()
INFO:pykeen.nn.message_passing:Inferred num_blocks=128 by GCD heuristic.
INFO:pykeen.nn.message_passing:Inferred num_blocks=128 by GCD heuristic.
INFO:pykeen.nn.message_passing:Inferred num_blocks=128 by GCD heuristic.
INFO:pykeen.nn.message_passing:Inferred num_blocks=128 by GCD heuristic.
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
  (fwd): BlockDecomposition()
  (bwd): BlockDecomposition()
  (self_loop): Linear(in_features=128, out_features=128, bias=False)
  (dropout): Dropout(p=0.4, inplace=False)
  (activation): LeakyReLU(negative_slope=0.01)
) has parameters, but no reset_parameters.
  (fwd): BlockDecomposition()
  (bwd): BlockDecomposition()
  (self_loop): Linear(in_features=128, out_features=128, bias=False)
  (dropout): Dropout(p=0.4, inplace=False)
) has parameters, 

## Risultati

In [12]:
best_params

{'model.embedding_dim': 128,
 'model.num_layers': 2,
 'model.use_bias': False,
 'model.activation': torch.nn.modules.activation.LeakyReLU,
 'model.edge_dropout': 0.0,
 'model.self_loop_dropout': 0.4,
 'model.edge_weighting': 'inverse_out_degree',
 'model.decomposition': 'block',
 'loss.margin': 0.7165466748379576,
 'optimizer.lr': 0.005598443410429674,
 'negative_sampler.num_negs_per_pos': 4}

In [ ]:
metric_dict = final_result.metric_results.to_dict()
output_file = "metriche_rgcn.json"
with open(output_file, "w") as f:
    json.dump(metric_dict, f, indent=4)  

print(f"Metriche salvate in {output_file}")

Metriche salvate in metriche_rgcn.json


In [14]:
print("MRR:", final_result.get_metric('mrr'))
print("MR:", final_result.get_metric('mr'))
print("Hits@1:", final_result.get_metric('hits@1'))
print("Hits@3:", final_result.get_metric('hits@3'))
print("Hits@10:", final_result.get_metric('hits@10'))

MRR: 0.13667838275432587
MR: 2727.36767578125
Hits@1: 0.09025341130604289
Hits@3: 0.14769330734243014
Hits@10: 0.22040285899935022
